In [ ]:
!pip install "transformers<5.0.0"
!pip install git+https://huggingface.co/Shubhamw11/Gemma-270M-TinyStories


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 156.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Cloning https://huggingface.co/Shubhamw11/Gemma-270M-TinyStories to /tmp/pip-req-build-jyjdti22
  Running command git clone --filter=blob:none --quiet https://huggingface.co/Shubhamw11/Gemma-270M-TinyStories /tmp/pip-req-build-jyjdti22
  Resolved https://huggingface.co/Shubhamw11/Gemma-270M-TinyStories to commit 99dfde33e347b525723f872b86733fcf7dfd2149
  Preparing metadata (setup.py) ... do

In [1]:
import torch
import numpy as np
import json
import math
from tqdm import tqdm
import tiktoken

from gemma3_tinystories import HFGemma3Model, Gemma3Config

In [2]:
!pip show transformers

Name: transformers
Version: 4.57.6
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: gemma3_tinystories, peft, sentence-transformers


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print(torch.cuda.get_device_name(0))

Using device: cuda
NVIDIA A100-SXM4-40GB


In [10]:
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

In [11]:
#define the device
config = Gemma3Config.from_pretrained("Shubhamw11/Gemma-270M-TinyStories")
hf_model = HFGemma3Model.from_pretrained(
    "Shubhamw11/Gemma-270M-TinyStories",
    config=config
)

model = hf_model.model if hasattr(hf_model, "model") else hf_model
tokenizer = tiktoken.get_encoding("gpt2")


device = "cuda" if torch.cuda.is_available() else "cpu"

input_text = "Once upon a time, there was a little"
context = torch.tensor(tokenizer.encode(input_text), dtype=torch.long).unsqueeze(0).to(device)
model.to(device)
response = model.generate(context, max_new_tokens=200, temperature=1.1, top_k=5)

print(tokenizer.decode(response.squeeze().tolist()))


Once upon a time, there was a little girl named Lily. She liked to play outside in the garden and watch the flowers grow. One day, she saw a bee buzzing around the flowers. The bee was very busy buzzing around the flowers.

"Hello bee, can you help me?" Lily asked.

The bee smiled and nodded its head. They watched as the bee flew around the flowers, enjoying the warm sun and buzzing bees.

"Thank you for helping me, little girl. You are very kind!" Lily said, giving her a big hug.

"You're welcome, Lily! We're glad you had fun!" The bee said.

Lily smiled and continued to gaze at the flowers and butterflies. She was happy that she could help her new friend and make her smile even more.Once upon a time, there was a little girl named Lily. She loved to play with her friends and explore the woods. One day, while she was wandering in the woods, she stumbled upon a


In [12]:
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze only the last 4 blocks + output layers
trainable_layers = ["blocks.14", "blocks.15", "blocks.16", "blocks.17", "final_norm", "out_head"]

for name, param in model.named_parameters():
    if any(name.startswith(layer) for layer in trainable_layers):
        param.requires_grad = True

# Step 3: Verify
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}  ({100*trainable/total:.1f}%)")
print(f"Frozen params:    {frozen:,}  ({100*frozen/total:.1f}%)")


Total params:     164,654,976
Trainable params: 54,459,648  (33.1%)
Frozen params:    110,195,328  (66.9%)


In [13]:
import json
import random

with open("negative_Tinystories_Filtered.json") as f:
    data = json.load(f)

# keep only valid indices
valid_indices = [
    i for i, x in enumerate(data)
    if x["original"] != x["modified"]
]

# shuffle indices (NOT text)
random.shuffle(valid_indices)

n = len(valid_indices)

train_split = int(0.7 * n)
val_split = int(0.85 * n)

train_indices = valid_indices[:train_split]
val_indices = valid_indices[train_split:val_split]
rlhf_indices = valid_indices[val_split:]

print("Train:", len(train_indices))
print("Val:", len(val_indices))
print("RLHF:", len(rlhf_indices))

Train: 8323
Val: 1783
RLHF: 1784


In [6]:
import json

with open("splits.json", "w") as f:
    json.dump({
        "train": train_indices,
        "val": val_indices,
        "rlhf": rlhf_indices
    }, f)

In [7]:
train_data_list = [data[i]["modified"] for i in train_indices]
val_data_list   = [data[i]["modified"] for i in val_indices]

In [14]:
def normalize(text):
    text = text.strip()
    if text.lower().startswith("once upon a time"):
        return text
    else:
        return "Once upon a time, " + text

# TRAIN
train_encoded = [
    tokenizer.encode(normalize(text))
    for text in train_data_list
]

# VAL
val_encoded = [
    tokenizer.encode(normalize(text))
    for text in val_data_list
]

print("Train encoded:", len(train_encoded))
print("Val encoded:", len(val_encoded))
print("Example length:", len(train_encoded[0]))

Train encoded: 8323
Val encoded: 1783
Example length: 78


In [15]:
import numpy as np

# TRAIN
train_tokens = []
for ids in train_encoded:
    train_tokens.extend(ids)

train_tokens = np.array(train_tokens, dtype=np.uint16)
train_tokens.tofile("TinyStories_SFT_train.bin")

# VAL
val_tokens = []
for ids in val_encoded:
    val_tokens.extend(ids)

val_tokens = np.array(val_tokens, dtype=np.uint16)
val_tokens.tofile("TinyStories_SFT_val.bin")

print("Train tokens:", len(train_tokens))
print("Val tokens:", len(val_tokens))

Train tokens: 1732601
Val tokens: 365761


In [16]:
train_data = np.memmap("TinyStories_SFT_train.bin", dtype=np.uint16, mode='r')
val_data   = np.memmap("TinyStories_SFT_val.bin", dtype=np.uint16, mode='r')

print("Train tokens:", len(train_data))
print("Val tokens:", len(val_data))


Train tokens: 1732601
Val tokens: 365761


In [17]:
# Batch Loader

block_size = 128
batch_size = 32

def get_train_batch():
    ix = torch.randint(len(train_data) - block_size, (batch_size,))

    x = torch.stack([
        torch.from_numpy(train_data[i:i+block_size].astype(np.int64))
        for i in ix
    ]).pin_memory().to(device, non_blocking=True)

    y = torch.stack([
        torch.from_numpy(train_data[i+1:i+1+block_size].astype(np.int64))
        for i in ix
    ]).pin_memory().to(device, non_blocking=True)

    return x, y


def get_val_batch():
    ix = torch.randint(len(val_data) - block_size, (batch_size,))

    x = torch.stack([
        torch.from_numpy(val_data[i:i+block_size].astype(np.int64))
        for i in ix
    ]).pin_memory().to(device, non_blocking=True)

    y = torch.stack([
        torch.from_numpy(val_data[i+1:i+1+block_size].astype(np.int64))
        for i in ix
    ]).pin_memory().to(device, non_blocking=True)

    return x, y

In [18]:
#Evaluation function
@torch.no_grad()
def estimate_loss(eval_iters=50):
    model.eval()

    losses = {"train": 0.0, "val": 0.0}

    # TRAIN LOSS
    for _ in range(eval_iters):
        x, y = get_train_batch()
        _, loss = model(x, y)
        losses["train"] += loss.item()

    # VAL LOSS
    for _ in range(eval_iters):
        x, y = get_val_batch()
        _, loss = model(x, y)
        losses["val"] += loss.item()

    # average
    losses["train"] /= eval_iters
    losses["val"] /= eval_iters

    model.train()

    return losses

In [19]:
losses = estimate_loss()
print(losses)

{'train': 2.0139080452919007, 'val': 1.9771785521507264}


In [8]:
learning_rate = 1e-5
max_iters = 1000
warmup_iters = 100
grad_clip = 1.0
weight_decay = 0.1
eval_interval = 50

patience = 5
no_improve_count = 0
best_val_loss = float('inf')
best_model_state = None


In [21]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    weight_decay=weight_decay
)

In [22]:
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * it / warmup_iters

    return learning_rate * 0.5 * (
        1 + math.cos(math.pi * (it - warmup_iters) / (max_iters - warmup_iters))
    )

print("Optimizer + scheduler ready")

Optimizer + scheduler ready


In [23]:
# SFT Training Loop
import time

train_losses = []
val_losses = []

model.train()

for iter in range(max_iters):
    lr = get_lr(iter)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    x, y = get_train_batch()
    logits, loss = model(x, y)

    model.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if iter % eval_interval == 0:
        losses = estimate_loss()

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve_count = 0
            print(f"✓ New best at iter {iter} | train {losses['train']:.4f} | val {losses['val']:.4f}")
        else:
            no_improve_count += 1
            print(f"  iter {iter} | train {losses['train']:.4f} | val {losses['val']:.4f} | no improve: {no_improve_count}/{patience}")
            if no_improve_count >= patience:
                print(f"Early stopping at iter {iter}")
                break

# Save best checkpoint
if best_model_state:
    torch.save(best_model_state, "gemma_sft_best.pt")
    print(f"Saved best model | val loss: {best_val_loss:.4f}")

iter 0 | train 2.0131 | val 1.9799 | time 8.51s | 8.514s/iter
iter 100 | train 1.9163 | val 1.9128 | time 8.52s | 0.084s/iter
iter 200 | train 1.8394 | val 1.8856 | time 8.52s | 0.042s/iter
iter 300 | train 1.7911 | val 1.8640 | time 8.53s | 0.028s/iter
iter 400 | train 1.7634 | val 1.8727 | time 8.53s | 0.021s/iter
iter 500 | train 1.7387 | val 1.8535 | time 8.53s | 0.017s/iter
iter 600 | train 1.7181 | val 1.8611 | time 8.52s | 0.014s/iter
iter 700 | train 1.7079 | val 1.8620 | time 8.52s | 0.012s/iter
iter 800 | train 1.6913 | val 1.8575 | time 8.53s | 0.011s/iter
iter 900 | train 1.6538 | val 1.8720 | time 8.52s | 0.009s/iter
iter 1000 | train 1.6584 | val 1.8763 | time 8.52s | 0.009s/iter
iter 1100 | train 1.6394 | val 1.8640 | time 8.52s | 0.008s/iter
iter 1200 | train 1.6337 | val 1.8815 | time 8.52s | 0.007s/iter
iter 1300 | train 1.5779 | val 1.8637 | time 8.52s | 0.007s/iter
iter 1400 | train 1.5972 | val 1.8869 | time 8.52s | 0.006s/iter
iter 1500 | train 1.5616 | val 1.8691

KeyboardInterrupt: 

In [ ]:
if best_model_state is not None:
    torch.save(best_model_state, "gemma_sft_best.pt")
    print(f"Best model saved with val loss: {best_val_loss:.4f}")